In [ ]:
!nvidia-smi
!nvcc --version

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Compute capability:", torch.cuda.get_device_capability(0))
    print("PyTorch CUDA:", torch.version.cuda)

Wed Aug 26 05:19:19 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   59C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import files

uploaded = files.upload()


print("Uploaded files:")
for name in uploaded.keys():
    print(" -", name)

Saving metagross_v2.zip to metagross_v2.zip
Uploaded files:
 - metagross_v2.zip


In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

uploaded_name = next(iter(uploaded.keys()))

work_dir = Path("/content/metagross")
extract_dir = Path("/content/metagross_extracted")

shutil.rmtree(work_dir, ignore_errors=True)
shutil.rmtree(extract_dir, ignore_errors=True)

extract_dir.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(uploaded_name, "r") as z:
    z.extractall(extract_dir)

# Find the directory containing setup.py
setup_files = list(extract_dir.rglob("setup.py"))

if not setup_files:
    raise FileNotFoundError("Could not find setup.py inside the ZIP.")

repo_root = setup_files[0].parent

shutil.copytree(repo_root, work_dir)

print("Repository:", work_dir)
print("Files:")
for p in sorted(work_dir.iterdir()):
    print(" ", p.name)

Repository: /content/metagross
Files:
  .gitignore
  LICENSE
  PATCH_NOTES.md
  README.md
  benchmarks
  csrc
  metagross
  metagross.png
  pyproject.toml
  setup.py
  tests


In [ ]:
from pathlib import Path

generate_file = Path("/content/metagross/metagross/generate.py")
attention_file = Path("/content/metagross/metagross/attention.py")

print("generate.py exists:", generate_file.exists())
print("attention.py exists:", attention_file.exists())

generate_text = generate_file.read_text()

print("\nChecking import:")
for line in generate_text.splitlines():
    if "paged_attention" in line and "import" in line:
        print(line)

print("\nChecking attention.py:")
attention_text = attention_file.read_text()

if "def paged_attention" in attention_text:
    print("paged_attention() FOUND")
else:
    raise RuntimeError("paged_attention() not found")

generate.py exists: True
attention.py exists: True

Checking import:
from .attention import paged_attention

Checking attention.py:
paged_attention() FOUND


In [ ]:
!pip install -U pip setuptools wheel
!pip install -U pytest numpy transformers datasets matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 6.2 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 34.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 56.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 27.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 51.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 30.0 MB/s  0:00:01
  Attempting uninstall: pytest
    Found existing installation: pytest 8.4.2
    Uninstalling pytest-8.4.2:
      Successfully uninstalled pytest-8.4.2
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: numpy
    Found existing installation: numpy 2.1.3
    Uninstalling numpy-2.1.3:
      Successfully uninstalled numpy-2.1.3
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.10.0
    Uninstalling matplotlib-3.10.0:
      Successfully uninstalled matplo

In [ ]:
import os

os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["PATH"] += ":/usr/local/cuda/bin"

# Tesla T4 = compute capability 7.5
os.environ["TORCH_CUDA_ARCH_LIST"] = "7.5"

# Prevent excessive parallel compilation memory usage
os.environ["MAX_JOBS"] = "2"

print("CUDA_HOME =", os.environ["CUDA_HOME"])
print("TORCH_CUDA_ARCH_LIST =", os.environ["TORCH_CUDA_ARCH_LIST"])
print("MAX_JOBS =", os.environ["MAX_JOBS"])

CUDA_HOME = /usr/local/cuda
TORCH_CUDA_ARCH_LIST = 7.5
MAX_JOBS = 2


In [ ]:
%cd /content/metagross

!pip uninstall -y metagross
!pip install -e . --no-build-isolation

/content/metagross
Obtaining file:///content/metagross
  Checking if build backend supports build_editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for metagross (pyproject.toml) ... done
  Created wheel for metagross: filename=metagross-0.0.1-0.editable-cp313-cp313-linux_x86_64.whl size=3629 sha256=ffbd2fe4d3af1bee0fafff4abda2c3e817d4b88a7e37e9b6a9a7b25b388ce176
  Stored in directory: /tmp/pip-ephem-wheel-cache-elkziv07/wheels/e1/33/6c/508527a85badf64d9801752665dfbc5763c0153b182af3f602
Successfully built metagross


In [ ]:
%cd /content/metagross

import torch
import metagross
from metagross import _C

print("Metagross version:", metagross.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

print("\nCompiled functions:")
for name in dir(_C):
    if not name.startswith("_"):
        print(" ", name)

/content/metagross
Metagross version: 0.0.1
CUDA available: True
GPU: Tesla T4

Compiled functions:
  add
  dequantize_int4_page
  dequantize_page
  paged_attention_committed
  quantize_fixed_scale
  quantize_int4_new_scale
  quantize_new_scale


In [ ]:
import torch
import metagross

device = "cuda"

a = torch.randn(1024, device=device, dtype=torch.float32)
b = torch.randn(1024, device=device, dtype=torch.float32)

out = metagross.sanity_add(a, b)

expected = a + b

print("Maximum error:", (out - expected).abs().max().item())
print("Correct:", torch.allclose(out, expected, atol=1e-6))

Maximum error: 0.0
Correct: True


In [ ]:
%cd /content/metagross

!pytest tests/test_sanity.py tests/test_block_allocator.py -v

/content/metagross
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-9.1.1, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/metagross
configfile: pyproject.toml
plugins: typeguard-4.6.0, anyio-4.14.2, langsmith-0.11.0
collected 9 items                                                              

tests/test_sanity.py::test_sanity_add_matches_torch PASSED               [ 11%]
tests/test_sanity.py::test_sanity_add_shape_mismatch_raises PASSED       [ 22%]
tests/test_block_allocator.py::test_allocate_returns_distinct_pages PASSED [ 33%]
tests/test_block_allocator.py::test_allocate_raises_when_exhausted PASSED [ 44%]
tests/test_block_allocator.py::test_free_makes_page_available_again PASSED [ 55%]
tests/test_block_allocator.py::test_double_free_raises PASSED            [ 66%]
tests/test_block_allocator.py::test_free_out_of_range_raises PASSED      [ 77%]
tests/test_block_allocator.py::test_n

In [ ]:
%cd /content/metagross

!pytest tests/test_quantize.py tests/test_int4.py -v

/content/metagross
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-9.1.1, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/metagross
configfile: pyproject.toml
plugins: typeguard-4.6.0, anyio-4.14.2, langsmith-0.11.0
collected 19 items                                                             

tests/test_quantize.py::TestQuantizeKernelAgainstReference::test_quantize_new_scale_matches_reference PASSED [  5%]
tests/test_quantize.py::TestQuantizeKernelAgainstReference::test_dequantize_recovers_within_half_step PASSED [ 10%]
tests/test_quantize.py::TestQuantizeKernelAgainstReference::test_fixed_scale_path_matches_reference_dequant PASSED [ 15%]
tests/test_quantize.py::TestQuantizeKernelAgainstReference::test_fixed_scale_clamps_out_of_range_values PASSED [ 21%]
tests/test_quantize.py::TestQuantizeKernelAgainstReference::test_offset_write_does_not_disturb_earlier_tokens PASSED [ 26%]
tests/

In [ ]:
%cd /content/metagross

!pytest tests/test_attention.py -v

/content/metagross
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-9.1.1, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/metagross
configfile: pyproject.toml
plugins: typeguard-4.6.0, anyio-4.14.2, langsmith-0.11.0
collected 10 items                                                             

tests/test_attention.py::TestPagedAttentionKernelDirectly::test_matches_reference_when_everything_is_committed PASSED [ 10%]
tests/test_attention.py::TestStagingContribution::test_matches_reference PASSED [ 20%]
tests/test_attention.py::TestStagingContribution::test_empty_staging_returns_identity PASSED [ 30%]
tests/test_attention.py::TestMergeUnnormalized::test_merging_with_identity_is_a_noop PASSED [ 40%]
tests/test_attention.py::TestMergeUnnormalized::test_merge_matches_reference_full_attention PASSED [ 50%]
tests/test_attention.py::TestPagedAttentionEndToEnd::test_matches_reference_with_mixe

In [ ]:
%cd /content/metagross

!pytest tests/ -v

/content/metagross
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-9.1.1, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/metagross
configfile: pyproject.toml
plugins: typeguard-4.6.0, anyio-4.14.2, langsmith-0.11.0
collected 194 items                                                            

tests/test_attention.py::TestPagedAttentionKernelDirectly::test_matches_reference_when_everything_is_committed PASSED [  0%]
tests/test_attention.py::TestStagingContribution::test_matches_reference PASSED [  1%]
tests/test_attention.py::TestStagingContribution::test_empty_staging_returns_identity PASSED [  1%]
tests/test_attention.py::TestMergeUnnormalized::test_merging_with_identity_is_a_noop PASSED [  2%]
tests/test_attention.py::TestMergeUnnormalized::test_merge_matches_reference_full_attention PASSED [  2%]
tests/test_attention.py::TestPagedAttentionEndToEnd::test_matches_reference_with_mixe

In [ ]:
import torch

from metagross.generate import load_gpt2

device = "cuda"

model, tokenizer = load_gpt2(device=device)

print("Model loaded")
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Device:", next(model.parameters()).device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model loaded
Parameters: 124439808
Device: cuda:0


In [ ]:
prompt = "The capital of France is"

In [ ]:
from metagross.generate import generate_metagross

text_meta, token_ids_meta, logits_meta, cache_meta = generate_metagross(
    model,
    tokenizer,
    prompt,
    max_new_tokens=20,
    page_size=16,
)

print("Metagross Phase 1 output:")
print(text_meta)

print("\nGenerated IDs:")
print(token_ids_meta)

print("\nCache sequence length:")
print(cache_meta.seq_len)

print("\nCache memory:")
print(cache_meta.memory_bytes(), "bytes")

Metagross Phase 1 output:
 the capital of the French Republic, and the capital of the Republic of France is the capital of the

Generated IDs:
[262, 3139, 286, 262, 4141, 2066, 11, 290, 262, 3139, 286, 262, 2066, 286, 4881, 318, 262, 3139, 286, 262]

Cache sequence length:
24

Cache memory:
1179648 bytes


In [ ]:
from metagross.generate import generate_baseline_hf

prompt = "The capital of France is"

text, token_ids, first_logits = generate_baseline_hf(
    model,
    tokenizer,
    prompt,
    max_new_tokens=20,
)

print("Baseline:")
print(text)
print(token_ids)

Baseline:
 the capital of the French Republic, and the capital of the French Republic is the capital of the French
[262, 3139, 286, 262, 4141, 2066, 11, 290, 262, 3139, 286, 262, 4141, 2066, 318, 262, 3139, 286, 262, 4141]


In [ ]:
print("Baseline:")
print(token_ids)

print("\nMetagross:")
print(token_ids_meta)

print("\nExact token match:")
print(token_ids == token_ids_meta)

print("\nBaseline text:")
print(text)

print("\nMetagross text:")
print(text_meta)

Baseline:
[262, 3139, 286, 262, 4141, 2066, 11, 290, 262, 3139, 286, 262, 4141, 2066, 318, 262, 3139, 286, 262, 4141]

Metagross:
[262, 3139, 286, 262, 4141, 2066, 11, 290, 262, 3139, 286, 262, 2066, 286, 4881, 318, 262, 3139, 286, 262]

Exact token match:
False

Baseline text:
 the capital of the French Republic, and the capital of the French Republic is the capital of the French

Metagross text:
 the capital of the French Republic, and the capital of the Republic of France is the capital of the


In [ ]:
  from metagross.generate import generate_metagross_fused

text_fused, token_ids_fused, logits_fused, cache_fused = generate_metagross_fused(
    model,
    tokenizer,
    prompt,
    max_new_tokens=20,
    page_size=16,
)

print("Metagross Fused output:")
print(text_fused)

print("\nGenerated IDs:")
print(token_ids_fused)

print("\nCache sequence length:")
print(cache_fused.seq_len)

print("\nCache memory:")
print(cache_fused.memory_bytes(), "bytes")

Metagross Fused output:
 the capital of the French Republic, and the capital of the Republic of France is the capital of the

Generated IDs:
[262, 3139, 286, 262, 4141, 2066, 11, 290, 262, 3139, 286, 262, 2066, 286, 4881, 318, 262, 3139, 286, 262]

Cache sequence length:
24

Cache memory:
1179648 bytes


In [ ]:
prompt = "The capital of France is"

text, token_ids, baseline_logits = generate_baseline_hf(
    model,
    tokenizer,
    prompt,
    max_new_tokens=20,
)

text_meta, token_ids_meta, meta_logits, cache_meta = generate_metagross(
    model,
    tokenizer,
    prompt,
    max_new_tokens=20,
    page_size=16,
)

print("BASELINE:")
print(text)

print("\nMETAGROSS:")
print(text_meta)

print("\nTOKEN MATCH:")
print(token_ids == token_ids_meta)

BASELINE:
 the capital of the French Republic, and the capital of the French Republic is the capital of the French

METAGROSS:
 the capital of the French Republic, and the capital of the Republic of France is the capital of the

TOKEN MATCH:
False


In [ ]:
print("=" * 70)
print("BASELINE")
print("=" * 70)
print(text)
print(token_ids)

print("\n" + "=" * 70)
print("METAGROSS PHASE 1")
print("=" * 70)
print(text_meta)
print(token_ids_meta)

print("\n" + "=" * 70)
print("METAGROSS PHASE 2 FUSED")
print("=" * 70)
print(text_fused)
print(token_ids_fused)

print("\n" + "=" * 70)
print("COMPARISON")
print("=" * 70)

print("Phase 1 == Baseline :", token_ids_meta == token_ids)
print("Phase 2 == Baseline :", token_ids_fused == token_ids)
print("Phase 1 == Phase 2  :", token_ids_meta == token_ids_fused)

BASELINE
 the capital of the French Republic, and the capital of the French Republic is the capital of the French
[262, 3139, 286, 262, 4141, 2066, 11, 290, 262, 3139, 286, 262, 4141, 2066, 318, 262, 3139, 286, 262, 4141]

METAGROSS PHASE 1
 the capital of the French Republic, and the capital of the Republic of France is the capital of the
[262, 3139, 286, 262, 4141, 2066, 11, 290, 262, 3139, 286, 262, 2066, 286, 4881, 318, 262, 3139, 286, 262]

METAGROSS PHASE 2 FUSED
 the capital of the French Republic, and the capital of the Republic of France is the capital of the
[262, 3139, 286, 262, 4141, 2066, 11, 290, 262, 3139, 286, 262, 2066, 286, 4881, 318, 262, 3139, 286, 262]

COMPARISON
Phase 1 == Baseline : False
Phase 2 == Baseline : False
Phase 1 == Phase 2  : True


In [ ]:
import torch

torch.cuda.synchronize()

allocated = torch.cuda.memory_allocated() / 1024**2
reserved = torch.cuda.memory_reserved() / 1024**2
peak = torch.cuda.max_memory_allocated() / 1024**2

print(f"Current allocated: {allocated:.2f} MB")
print(f"Current reserved : {reserved:.2f} MB")
print(f"Peak allocated   : {peak:.2f} MB")

Current allocated: 259.94 MB
Current reserved : 284.00 MB
Peak allocated   : 259.99 MB


In [ ]:
import inspect
import metagross
import metagross.generate
import metagross.attention

print("metagross:")
print(metagross.__file__)

print("\nmetagross.generate:")
print(metagross.generate.__file__)

print("\nmetagross.attention:")
print(metagross.attention.__file__)

print("\nPaged attention function:")
print(inspect.getsource(metagross.attention.paged_attention)[:1000])

metagross:
/content/metagross/metagross/__init__.py

metagross.generate:
/content/metagross/metagross/generate.py

metagross.attention:
/content/metagross/metagross/attention.py

Paged attention function:
def paged_attention(cache: PagedKVCache, layer_idx: int, query: torch.Tensor, scale: float | None = None) -> torch.Tensor:
    """
    Full causal attention output for ONE new query token at `layer_idx`,
    combining the fused kernel's result over committed pages with
    staging's own contribution, via the verified merge above.

    CALLING CONVENTION -- read before wiring this up: the current step's
    K and V must already be appended to `cache` (via `cache.append(...)`)
    BEFORE calling this. Causal self-attention includes the current
    position attending to its own key/value, not just everything before
    it; this function has no separate "and also attend to the current
    token" path; it only ever reads whatever's already in the cache. Call
    it before appending and the